[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_82_Phase9_Capstone_Shipping_Orchestra.ipynb)

# Lesson 82 — Phase 9 Capstone: Shipping `orchestra`

**Phase 9, Lesson 6 of 6 — the capstone. This closes Phase 9.**

Over the last five lessons you built five modules:

- **L77 `core`** — `Message` + `Agent`: the control-flow substrate.
- **L78 `blackboard`** — a versioned, concurrency-safe shared workspace.
- **L79 `router`** — classify a task, dispatch to the best specialist, escalate.
- **L80 `planner`** — decompose a goal into an ordered DAG of sub-tasks.
- **L81 `reliability`** — retries, timeouts, circuit breakers, partial-failure.

Each one passed its own tests. **But five modules that each pass their own tests
are not a library.** A library is those modules proven to *compose*, packaged
behind one API, and shipped so a stranger can `pip install` it and use it.

> **The one idea of this capstone:** the load-bearing risk isn't any single
> module — it's that modules validated *in isolation* silently diverge when you
> wire them together. So the heart of this lesson is a single **integration
> test** that drives all five pillars at once, followed by a real build →
> `twine check` → fresh-wheel install, then launch-day materials for a **fourth**
> open-source artifact alongside `paper-distiller`, `agent-bench`, and `agent-obs`.

Same rules as always: no API key, everything deterministic, every claim ends in
an `assert`.

## Phase 9 in one table

| Lesson | Module | The problem it solved | Key primitive |
|---|---|---|---|
| L77 | `core` | how do agents even talk? | `Message`, `Agent` |
| L78 | `blackboard` | point-to-point wiring is O(N²) | `Blackboard` (versioned, atomic) |
| L79 | `router` | *which* agent should do this task? | `Classifier` + `Router` (gate, handoff) |
| L80 | `planner` | a goal is several ordered sub-tasks | `Plan`, `topo_order`, `layers`, `Planner` |
| L81 | `reliability` | steps fail; don't crash or hammer | `retry`, `Reliable`, `CircuitBreaker`, `run_layer` |
| **L82** | **the package** | **do they actually compose + ship?** | **`ResilientOrchestrator` + a wheel** |

The router answers *who*. The planner answers *what, in what order*. The
reliability layer answers *what happens when a step fails*. The blackboard is
*where shared state lives*. The capstone answers the only question that a
portfolio reviewer actually cares about: **does it all work together, and can I
install it?**

In [ ]:
# ── Setup: write the whole orchestra-agents project to disk ─────────────────
# On Google Colab BASE = "/content" (its working directory). Everything the
# package needs — the five modules, __init__.py, pyproject.toml, README.md — is
# embedded as base64 so no quote or docstring inside them can ever collide with
# a notebook cell delimiter (the pattern we settled on back in L77).
import base64, os, sys, subprocess, importlib

BASE = "/content"                                  # Colab's working directory
PROJ = os.path.join(BASE, "orchestra-agents")      # the project root we build from
os.makedirs(os.path.join(PROJ, "orchestra"), exist_ok=True)

_FILES = {
    "orchestra/core.py": "IyBvcmNoZXN0cmEvY29yZS5weSAgLS0gIG1lc3NhZ2UgKyBhZ2VudCBwcmltaXRpdmVzIChmcm9tIExlc3NvbiA3NykKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZQoKQGRhdGFjbGFzcwpjbGFzcyBNZXNzYWdlOgogICAgc2VuZGVyOiBzdHIKICAgIHJlY2lwaWVudDogc3RyCiAgICBraW5kOiBzdHIgICAgICAgICAgICAgICAgICMgInRhc2siIHwgInJlc3VsdCIgfCAiaGFuZG9mZiIgfCAuLi4KICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKY2xhc3MgQWdlbnQ6CiAgICAjIEFuIGFnZW50ID0gb25lIGNhbGxhYmxlICJicmFpbiIgYmVoaW5kIGEgbmFtZS4gYmFja2VuZChuYW1lLCBjb250ZW50KSAtPiAob3V0cHV0LCB0b2tlbnMpCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyLCByb2xlOiBzdHIsIGJhY2tlbmQ6IENhbGxhYmxlKToKICAgICAgICBzZWxmLm5hbWUgPSBuYW1lCiAgICAgICAgc2VsZi5yb2xlID0gcm9sZQogICAgICAgIHNlbGYuYmFja2VuZCA9IGJhY2tlbmQKICAgICAgICBzZWxmLmNhbGxzID0gMAogICAgZGVmIGFjdChzZWxmLCBtc2c6ICJNZXNzYWdlIikgLT4gIk1lc3NhZ2UiOgogICAgICAgIHNlbGYuY2FsbHMgKz0gMQogICAgICAgIG91dCwgdG9rZW5zID0gc2VsZi5iYWNrZW5kKHNlbGYubmFtZSwgbXNnLmNvbnRlbnQpCiAgICAgICAgcmV0dXJuIE1lc3NhZ2Uoc2VuZGVyPXNlbGYubmFtZSwgcmVjaXBpZW50PW1zZy5zZW5kZXIsCiAgICAgICAgICAgICAgICAgICAgICAga2luZD0icmVzdWx0IiwgY29udGVudD1vdXQsIG1ldGE9eyJ0b2tlbnMiOiB0b2tlbnN9KQo=",
    "orchestra/blackboard.py": "IiIib3JjaGVzdHJhLmJsYWNrYm9hcmQg4oCUIGEgc2hhcmVkIHN0cnVjdHVyZWQgd29ya3NwYWNlIGZvciBtYW55IGFnZW50cy4KCkluIG9yY2hlc3RyYS5jb3JlIChMZXNzb24gNzcpIGFnZW50cyBwYXNzIE1lc3NhZ2VzIHBvaW50LXRvLXBvaW50LiBUaGF0IGlzCmZpbmUgZm9yIGEgZml4ZWQgaGFuZGZ1bCBvZiBhZ2VudHMuIFdoZW4gbWFueSBhZ2VudHMgbXVzdCBzaGFyZSBldm9sdmluZwpzdGF0ZSwgcG9pbnQtdG8tcG9pbnQgd2lyaW5nIGlzIE8oTl4yKTogZXZlcnkgbmV3IGFnZW50IG1lYW5zIHJld2lyaW5nIHRoZQpyZXN0LiBUaGUgYmxhY2tib2FyZCBmbGlwcyBpdDogYWdlbnRzIG5ldmVyIHRhbGsgdG8gZWFjaCBvdGhlci4gVGhleSByZWFkCmZyb20gYW5kIHdyaXRlIHRvIE9ORSBzaGFyZWQsIHN0cnVjdHVyZWQsIGNvbmN1cnJlbmN5LXNhZmUgd29ya3NwYWNlLiBUaGUKbmV3IGhhcmQgcHJvYmxlbSB0aGlzIGNyZWF0ZXMgaXMgQ09OU0lTVEVOQ1kgdW5kZXIgY29uY3VycmVudCB3cml0ZXMsIHdoaWNoCmlzIHdoeSBCbGFja2JvYXJkIGlzIGJ1aWx0IGFyb3VuZCB2ZXJzaW9uZWQsIGF0b21pYyBvcGVyYXRpb25zLgoiIiIKaW1wb3J0IHRocmVhZGluZwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIENhbGxhYmxlLCBPcHRpb25hbAoKCmNsYXNzIEJsYWNrYm9hcmQ6CiAgICAiIiJBIHRocmVhZC1zYWZlIGtleS92YWx1ZSB3b3Jrc3BhY2Ugd2l0aCBwZXIta2V5IHZlcnNpb25zLgoKICAgIEV2ZXJ5IHN1Y2Nlc3NmdWwgd3JpdGUgYnVtcHMgdGhhdCBrZXkncyB2ZXJzaW9uLiBWZXJzaW9ucyBhcmUgd2hhdCBtYWtlCiAgICBvcHRpbWlzdGljIGNvbmN1cnJlbmN5IChjb21wYXJlLWFuZC1zZXQpIHBvc3NpYmxlOiBhbiBhZ2VudCBjYW4gcHJvdmUgaXQKICAgIGlzIHdyaXRpbmcgb24gdG9wIG9mIHRoZSBzdGF0ZSBpdCBhY3R1YWxseSByZWFkLCBub3QgYSBzdGFsZSBjb3B5LgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYuX2RhdGEgPSB7fQogICAgICAgIHNlbGYuX3ZlcnNpb25zID0ge30KICAgICAgICBzZWxmLl9sb2NrID0gdGhyZWFkaW5nLlJMb2NrKCkKCiAgICBkZWYgZ2V0KHNlbGYsIGtleSwgZGVmYXVsdD1Ob25lKToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9kYXRhLmdldChrZXksIGRlZmF1bHQpCgogICAgZGVmIHZlcnNpb24oc2VsZiwga2V5KToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl92ZXJzaW9ucy5nZXQoa2V5LCAwKQoKICAgIGRlZiBnZXRfdmVyc2lvbmVkKHNlbGYsIGtleSwgZGVmYXVsdD1Ob25lKToKICAgICAgICAiIiJSZXR1cm4gKHZhbHVlLCB2ZXJzaW9uKSBhdG9taWNhbGx5IOKAlCB0aGUgcGFpciBDQVMgbmVlZHMuIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZGF0YS5nZXQoa2V5LCBkZWZhdWx0KSwgc2VsZi5fdmVyc2lvbnMuZ2V0KGtleSwgMCkKCiAgICBkZWYgc2V0KHNlbGYsIGtleSwgdmFsdWUpOgogICAgICAgICIiIlVuY29uZGl0aW9uYWwgd3JpdGUuIEJ1bXBzIHRoZSB2ZXJzaW9uLiBOT1Qgc2FmZSBmb3IKICAgICAgICByZWFkLW1vZGlmeS13cml0ZSByYWNlcyBvbiBpdHMgb3duIOKAlCB0aGF0IGlzIHdoYXQgdXBkYXRlL0NBUyBmaXguIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLl9kYXRhW2tleV0gPSB2YWx1ZQogICAgICAgICAgICBzZWxmLl92ZXJzaW9uc1trZXldID0gc2VsZi5fdmVyc2lvbnMuZ2V0KGtleSwgMCkgKyAxCiAgICAgICAgICAgIHJldHVybiBzZWxmLl92ZXJzaW9uc1trZXldCgogICAgZGVmIHVwZGF0ZShzZWxmLCBrZXksIGZuLCBkZWZhdWx0PU5vbmUpOgogICAgICAgICIiIkF0b21pYyByZWFkLW1vZGlmeS13cml0ZTogZm4ob2xkX3ZhbHVlKSAtPiBuZXdfdmFsdWUsIGV4ZWN1dGVkCiAgICAgICAgd2l0aCB0aGUgbG9jayBoZWxkIGZvciB0aGUgV0hPTEUgb3BlcmF0aW9uLiBUaGlzIGlzIHRoZSBwZXNzaW1pc3RpYwogICAgICAgIGZpeCBmb3IgbG9zdCB1cGRhdGVzLiIiIgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgb2xkID0gc2VsZi5fZGF0YS5nZXQoa2V5LCBkZWZhdWx0KQogICAgICAgICAgICBuZXcgPSBmbihvbGQpCiAgICAgICAgICAgIHNlbGYuX2RhdGFba2V5XSA9IG5ldwogICAgICAgICAgICBzZWxmLl92ZXJzaW9uc1trZXldID0gc2VsZi5fdmVyc2lvbnMuZ2V0KGtleSwgMCkgKyAxCiAgICAgICAgICAgIHJldHVybiBuZXcKCiAgICBkZWYgY29tcGFyZV9hbmRfc2V0KHNlbGYsIGtleSwgZXhwZWN0ZWRfdmVyc2lvbiwgdmFsdWUpOgogICAgICAgICIiIk9wdGltaXN0aWMgd3JpdGU6IG9ubHkgc3VjY2VlZHMgaWYgdGhlIGtleSdzIHZlcnNpb24gc3RpbGwgZXF1YWxzCiAgICAgICAgZXhwZWN0ZWRfdmVyc2lvbi4gUmV0dXJucyBUcnVlIG9uIHN1Y2Nlc3MsIEZhbHNlIGlmIHNvbWVvbmUgZWxzZSB3cm90ZQogICAgICAgIGZpcnN0ICh0aGUgY2FsbGVyIHNob3VsZCByZS1yZWFkIGFuZCByZXRyeSkuIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBpZiBzZWxmLl92ZXJzaW9ucy5nZXQoa2V5LCAwKSAhPSBleHBlY3RlZF92ZXJzaW9uOgogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgICAgIHNlbGYuX2RhdGFba2V5XSA9IHZhbHVlCiAgICAgICAgICAgIHNlbGYuX3ZlcnNpb25zW2tleV0gPSBleHBlY3RlZF92ZXJzaW9uICsgMQogICAgICAgICAgICByZXR1cm4gVHJ1ZQoKICAgIGRlZiBoYXMoc2VsZiwga2V5KToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBrZXkgaW4gc2VsZi5fZGF0YQoKICAgIGRlZiBrZXlzKHNlbGYpOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgcmV0dXJuIHNldChzZWxmLl9kYXRhLmtleXMoKSkKCiAgICBkZWYgc25hcHNob3Qoc2VsZik6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByZXR1cm4gZGljdChzZWxmLl9kYXRhKQoKCkBkYXRhY2xhc3MKY2xhc3MgS25vd2xlZGdlU291cmNlOgogICAgIiIiQSBrbm93bGVkZ2Ugc291cmNlIChhZ2VudCkgZGVjbGFyZXMgV0hFTiBpdCBtYXkgZmlyZSAodHJpZ2dlcikgYW5kCiAgICBXSEFUIGl0IGRvZXMgKHJ1bikuIEl0IHJlYWRzL3dyaXRlcyBvbmx5IHRoZSBibGFja2JvYXJkIOKAlCBuZXZlciBhbm90aGVyCiAgICBhZ2VudCBkaXJlY3RseS4gQ29udHJvbCBpcyBkYXRhLWRyaXZlbjogYSBzb3VyY2UgZmlyZXMgd2hlbiBpdHMKICAgIHByZWNvbmRpdGlvbnMgYXBwZWFyIG9uIHRoZSBib2FyZC4iIiIKICAgIG5hbWU6IHN0cgogICAgdHJpZ2dlcjogQ2FsbGFibGVbW0JsYWNrYm9hcmRdLCBib29sXQogICAgcnVuOiBDYWxsYWJsZVtbQmxhY2tib2FyZF0sIE5vbmVdCgoKZGVmIHJ1bl91bnRpbF9xdWllc2NlbnQoYmIsIHNvdXJjZXMsIG1heF9yb3VuZHM9MTAwKToKICAgICIiIkRhdGEtZHJpdmVuIGNvbnRyb2xsZXI6IHJlcGVhdGVkbHkgcnVuIGV2ZXJ5IHNvdXJjZSB3aG9zZSB0cmlnZ2VyCiAgICBmaXJlcywgdW50aWwgYSBmdWxsIHBhc3MgZmlyZXMgbm90aGluZyAocXVpZXNjZW5jZSkgb3IgbWF4X3JvdW5kcyBpcyBoaXQuCiAgICBSZXR1cm5zIChyb3VuZHMsIGZpcmVkKSB3aGVyZSBmaXJlZCBpcyB0aGUgb3JkZXJlZCBsaXN0IG9mIHNvdXJjZSBuYW1lcwogICAgdGhhdCByYW4uIG1heF9yb3VuZHMgaXMgdGhlIGd1YXJkIGFnYWluc3QgYSBub24tdGVybWluYXRpbmcgYm9hcmQuIiIiCiAgICByb3VuZHMgPSAwCiAgICBmaXJlZCA9IFtdCiAgICBmb3IgXyBpbiByYW5nZShtYXhfcm91bmRzKToKICAgICAgICByb3VuZHMgKz0gMQogICAgICAgIGZpcmVkX3RoaXNfcm91bmQgPSBGYWxzZQogICAgICAgIGZvciBrcyBpbiBzb3VyY2VzOgogICAgICAgICAgICBpZiBrcy50cmlnZ2VyKGJiKToKICAgICAgICAgICAgICAgIGtzLnJ1bihiYikKICAgICAgICAgICAgICAgIGZpcmVkLmFwcGVuZChrcy5uYW1lKQogICAgICAgICAgICAgICAgZmlyZWRfdGhpc19yb3VuZCA9IFRydWUKICAgICAgICBpZiBub3QgZmlyZWRfdGhpc19yb3VuZDoKICAgICAgICAgICAgYnJlYWsKICAgIHJldHVybiByb3VuZHMsIGZpcmVkCg==",
    "orchestra/router.py": "IyBvcmNoZXN0cmEvcm91dGVyLnB5ICAtLSAgYSByb3V0ZXIgdGhhdCBjbGFzc2lmaWVzIGVhY2ggdGFzayBhbmQgaGFuZHMgaXQKIyB0byB0aGUgYmVzdCBzcGVjaWFsaXN0LCBlc2NhbGF0ZXMgd2hlbiB1bnN1cmUsIGFuZCBzdXJ2aXZlcyBoYW5kb2ZmIGxvb3BzLgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgUm91dGU6CiAgICBjYXRlZ29yeTogT3B0aW9uYWxbc3RyXQogICAgY29uZmlkZW5jZTogZmxvYXQKCmNsYXNzIENsYXNzaWZpZXI6CiAgICAjIENoZWFwIGtleXdvcmQgY2xhc3NpZmllci4gSW4gcHJvZHVjdGlvbiB0aGlzIHdvdWxkIGJlIGFuIGVtYmVkZGluZyBtb2RlbAogICAgIyBvciBhIHNtYWxsIExMTTsgdGhlIGludGVyZmFjZSAodGV4dCAtPiBSb3V0ZSkgaXMgd2hhdCBtYXR0ZXJzLgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGtleXdvcmRzOiBEaWN0W3N0ciwgbGlzdF0pOgogICAgICAgIHNlbGYua2V5d29yZHMgPSBrZXl3b3JkcwogICAgICAgIHNlbGYuY2FsbHMgPSAwCiAgICBkZWYgY2xhc3NpZnkoc2VsZiwgdGV4dDogc3RyKSAtPiBSb3V0ZToKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICB0ID0gdGV4dC5sb3dlcigpCiAgICAgICAgc2NvcmVzID0ge30KICAgICAgICBmb3IgY2F0LCBrd3MgaW4gc2VsZi5rZXl3b3Jkcy5pdGVtcygpOgogICAgICAgICAgICBoaXRzID0gc3VtKDEgZm9yIGsgaW4ga3dzIGlmIGsgaW4gdCkKICAgICAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgICAgIHNjb3Jlc1tjYXRdID0gaGl0cwogICAgICAgIGlmIG5vdCBzY29yZXM6CiAgICAgICAgICAgIHJldHVybiBSb3V0ZShOb25lLCAwLjApICAgICAgICAgICAgICAjIG5vdGhpbmcgbWF0Y2hlZCAtPiBlc2NhbGF0ZQogICAgICAgIHRvdGFsID0gc3VtKHNjb3Jlcy52YWx1ZXMoKSkKICAgICAgICBiZXN0X2NhdCwgYmVzdF9oaXRzID0gc29ydGVkKHNjb3Jlcy5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoLWt2WzFdLCBrdlswXSkpWzBdCiAgICAgICAgcmV0dXJuIFJvdXRlKGJlc3RfY2F0LCBiZXN0X2hpdHMgLyB0b3RhbCkKCmNsYXNzIFJvdXRlcjoKICAgICMgSG9sZHMgc3BlY2lhbGlzdHMgKGNhdGVnb3J5IC0+IEFnZW50KSwgYSBnZW5lcmFsaXN0IGZhbGxiYWNrLCBhbmQgdGhlCiAgICAjIHBvbGljeTogZ2F0ZSBvbiBjb25maWRlbmNlLCBkaXNwYXRjaCwgZm9sbG93IGhhbmRvZmZzLCBjYXAgdGhlIGhvcHMuCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2xhc3NpZmllciwgc3BlY2lhbGlzdHMsIGdlbmVyYWxpc3QsCiAgICAgICAgICAgICAgICAgdGhyZXNob2xkPTAuNiwgaG9wX2NhcD0zLCBjbGFzc2lmeV9jb3N0PTIpOgogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IGNsYXNzaWZpZXIKICAgICAgICBzZWxmLnNwZWNpYWxpc3RzID0gZGljdChzcGVjaWFsaXN0cykKICAgICAgICBzZWxmLmdlbmVyYWxpc3QgPSBnZW5lcmFsaXN0CiAgICAgICAgc2VsZi50aHJlc2hvbGQgPSB0aHJlc2hvbGQKICAgICAgICBzZWxmLmhvcF9jYXAgPSBob3BfY2FwCiAgICAgICAgc2VsZi5jbGFzc2lmeV9jb3N0ID0gY2xhc3NpZnlfY29zdAoKICAgIGRlZiBfZXNjYWxhdGUoc2VsZiwgdGFzaywgdHJhY2UsIHRva2VucywgcmVhc29uKToKICAgICAgICBob3BzID0gbGVuKHRyYWNlKQogICAgICAgIG0gPSBzZWxmLmdlbmVyYWxpc3QuYWN0KE1lc3NhZ2UoInJvdXRlciIsIHNlbGYuZ2VuZXJhbGlzdC5uYW1lLCAidGFzayIsIHRhc2spKQogICAgICAgIHRyYWNlLmFwcGVuZChzZWxmLmdlbmVyYWxpc3QubmFtZSkKICAgICAgICB0b2tlbnMgKz0gbS5tZXRhLmdldCgidG9rZW5zIiwgMCkKICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywgInRyYWNlIjogdHJhY2UsCiAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6IHJlYXNvbiwgImhvcHMiOiBob3BzfQoKICAgIGRlZiBkaXNwYXRjaChzZWxmLCB0YXNrKToKICAgICAgICB0cmFjZSA9IFtdCiAgICAgICAgdG9rZW5zID0gc2VsZi5jbGFzc2lmeV9jb3N0CiAgICAgICAgcm91dGUgPSBzZWxmLmNsYXNzaWZpZXIuY2xhc3NpZnkodGFza1sidGV4dCJdKQogICAgICAgIGlmIHJvdXRlLmNhdGVnb3J5IGlzIE5vbmUgb3Igcm91dGUuY29uZmlkZW5jZSA8IHNlbGYudGhyZXNob2xkOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgImxvd19jb25maWRlbmNlIikKICAgICAgICBjdXJyZW50ID0gcm91dGUuY2F0ZWdvcnkKICAgICAgICBob3BzID0gMAogICAgICAgIHdoaWxlIGhvcHMgPCBzZWxmLmhvcF9jYXA6CiAgICAgICAgICAgIGhvcHMgKz0gMQogICAgICAgICAgICBpZiBjdXJyZW50IG5vdCBpbiBzZWxmLnNwZWNpYWxpc3RzOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VzY2FsYXRlKHRhc2ssIHRyYWNlLCB0b2tlbnMsICJ1bmtub3duX2NhdGVnb3J5IikKICAgICAgICAgICAgYWdlbnQgPSBzZWxmLnNwZWNpYWxpc3RzW2N1cnJlbnRdCiAgICAgICAgICAgIG0gPSBhZ2VudC5hY3QoTWVzc2FnZSgicm91dGVyIiwgYWdlbnQubmFtZSwgInRhc2siLCB0YXNrKSkKICAgICAgICAgICAgdHJhY2UuYXBwZW5kKGFnZW50Lm5hbWUpCiAgICAgICAgICAgIHRva2VucyArPSBtLm1ldGEuZ2V0KCJ0b2tlbnMiLCAwKQogICAgICAgICAgICB0YXJnZXQgPSBtLmNvbnRlbnQuZ2V0KCJoYW5kb2ZmIikKICAgICAgICAgICAgaWYgdGFyZ2V0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywKICAgICAgICAgICAgICAgICAgICAgICAgInRyYWNlIjogdHJhY2UsICJlc2NhbGF0ZWQiOiBGYWxzZSwgInJlYXNvbiI6ICJyb3V0ZWQiLCAiaG9wcyI6IGhvcHN9CiAgICAgICAgICAgIGN1cnJlbnQgPSB0YXJnZXQKICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgImhvcF9jYXAiKQo=",
    "orchestra/planner.py": "IyBvcmNoZXN0cmEvcGxhbm5lci5weSAgLS0gIGRlY29tcG9zZSBhIGdvYWwgaW50byBhbiBvcmRlcmVkIHBsYW4gKGEgREFHIG9mCiMgc3ViLXRhc2tzKSwgZGlzcGF0Y2ggZWFjaCBzdWItdGFzayB0aHJvdWdoIHRoZSBMZXNzb24tNzkgUm91dGVyLCB0aHJlYWQgZWFjaAojIGRlcGVuZGVuY3kncyBvdXRwdXQgaW50byB0aGUgdGFza3MgdGhhdCBkZXBlbmQgb24gaXQsIGFuZCBhc3NlbWJsZSBhIHJlc3VsdC4KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIE9wdGlvbmFsLCBMaXN0LCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgU3RlcDoKICAgIGlkOiBzdHIKICAgIHRleHQ6IHN0cgogICAgZGVwczogTGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpICAgIyBpZHMgdGhpcyBzdGVwIHdhaXRzIG9uCiAgICBwYXlsb2FkOiBkaWN0ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpICAgICAgIyB0aGUgYXRvbWljIHRhc2sgZmllbGRzCgpAZGF0YWNsYXNzCmNsYXNzIFBsYW46CiAgICBnb2FsOiBzdHIKICAgIHN0ZXBzOiBMaXN0W1N0ZXBdCiAgICBkZWYgYnlfaWQoc2VsZikgLT4gRGljdFtzdHIsICJTdGVwIl06CiAgICAgICAgcmV0dXJuIHtzLmlkOiBzIGZvciBzIGluIHNlbGYuc3RlcHN9CgpkZWYgdG9wb19vcmRlcihzdGVwczogTGlzdFtTdGVwXSkgLT4gTGlzdFtzdHJdOgogICAgIyBLYWhuJ3MgYWxnb3JpdGhtLiBSYWlzZXMgVmFsdWVFcnJvciBvbiBhIG1pc3NpbmcgZGVwIG9yIGEgY3ljbGUuCiAgICBpZHMgPSB7cy5pZCBmb3IgcyBpbiBzdGVwc30KICAgIGluZGVnID0ge3MuaWQ6IDAgZm9yIHMgaW4gc3RlcHN9CiAgICBhZGogPSB7cy5pZDogW10gZm9yIHMgaW4gc3RlcHN9CiAgICBmb3IgcyBpbiBzdGVwczoKICAgICAgICBmb3IgZCBpbiBzLmRlcHM6CiAgICAgICAgICAgIGlmIGQgbm90IGluIGlkczoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInN0ZXAgJXIgZGVwZW5kcyBvbiB1bmtub3duIHN0ZXAgJXIiICUgKHMuaWQsIGQpKQogICAgICAgICAgICBhZGpbZF0uYXBwZW5kKHMuaWQpCiAgICAgICAgICAgIGluZGVnW3MuaWRdICs9IDEKICAgIHJlYWR5ID0gc29ydGVkKFtpIGZvciBpIGluIGluZGVnIGlmIGluZGVnW2ldID09IDBdKSAgICMgZGV0ZXJtaW5pc3RpYyBvcmRlcgogICAgb3JkZXIgPSBbXQogICAgd2hpbGUgcmVhZHk6CiAgICAgICAgbiA9IHJlYWR5LnBvcCgwKQogICAgICAgIG9yZGVyLmFwcGVuZChuKQogICAgICAgIGZvciBtIGluIGFkaltuXToKICAgICAgICAgICAgaW5kZWdbbV0gLT0gMQogICAgICAgICAgICBpZiBpbmRlZ1ttXSA9PSAwOgogICAgICAgICAgICAgICAgcmVhZHkuYXBwZW5kKG0pCiAgICAgICAgcmVhZHkuc29ydCgpCiAgICBpZiBsZW4ob3JkZXIpICE9IGxlbihzdGVwcyk6CiAgICAgICAgc3R1Y2sgPSBsZW4oc3RlcHMpIC0gbGVuKG9yZGVyKQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBsYW4gaGFzIGEgY3ljbGU7ICVkIG9mICVkIHN0ZXBzIG5ldmVyIGJlY2FtZSByZWFkeSIKICAgICAgICAgICAgICAgICAgICAgICAgICUgKHN0dWNrLCBsZW4oc3RlcHMpKSkKICAgIHJldHVybiBvcmRlcgoKZGVmIGxheWVycyhzdGVwczogTGlzdFtTdGVwXSkgLT4gTGlzdFtMaXN0W3N0cl1dOgogICAgIyBHcm91cCBzdGVwcyBpbnRvIGRlcGVuZGVuY3kgbGF5ZXJzLiBFdmVyeSBzdGVwIGluIGEgbGF5ZXIgY2FuIHJ1biBpbgogICAgIyBwYXJhbGxlbDsgbGF5ZXIgayBkZXBlbmRzIG9ubHkgb24gbGF5ZXJzIDwgay4gVmFsaWRhdGVzIChyYWlzZXMgb24gY3ljbGUpLgogICAgb3JkZXIgPSB0b3BvX29yZGVyKHN0ZXBzKQogICAgYnlfaWQgPSB7cy5pZDogcyBmb3IgcyBpbiBzdGVwc30KICAgIGRlcHRoID0ge30KICAgIGZvciBzaWQgaW4gb3JkZXI6CiAgICAgICAgZHMgPSBieV9pZFtzaWRdLmRlcHMKICAgICAgICBkZXB0aFtzaWRdID0gMCBpZiBub3QgZHMgZWxzZSAxICsgbWF4KGRlcHRoW2RdIGZvciBkIGluIGRzKQogICAgb3V0ID0gW10KICAgIGZvciBzaWQgaW4gb3JkZXI6CiAgICAgICAgZCA9IGRlcHRoW3NpZF0KICAgICAgICB3aGlsZSBsZW4ob3V0KSA8PSBkOgogICAgICAgICAgICBvdXQuYXBwZW5kKFtdKQogICAgICAgIG91dFtkXS5hcHBlbmQoc2lkKQogICAgcmV0dXJuIFtzb3J0ZWQobCkgZm9yIGwgaW4gb3V0XQoKY2xhc3MgUGxhbm5lcjoKICAgICMgZGVjb21wb3NlIC0+IG9yZGVyIC0+IGRpc3BhdGNoIGVhY2ggdmlhIHRoZSBSb3V0ZXIgKHRocmVhZGluZyBkZXBlbmRlbmN5CiAgICAjIG91dHB1dHMgZm9yd2FyZCkgLT4gdmFsaWRhdGUgZWFjaCByZXN1bHQgYW5kIFJFUExBTiBmYWlsdXJlcyBvbiB0aGUKICAgICMgZ2VuZXJhbGlzdCAtPiBhc3NlbWJsZS4gVGhlIFJvdXRlciBhbnN3ZXJzICJ3aG8iOyB0aGUgUGxhbm5lciBhbnN3ZXJzCiAgICAjICJ3aGF0IGFyZSB0aGUgdGFza3MsIGluIHdoYXQgb3JkZXIsIGFuZCBkaWQgZWFjaCBvbmUgYWN0dWFsbHkgc3VjY2VlZCIuCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGVjb21wb3NlOiBDYWxsYWJsZSwgcm91dGVyLAogICAgICAgICAgICAgICAgIHZhbGlkYXRlOiBPcHRpb25hbFtDYWxsYWJsZV0gPSBOb25lKToKICAgICAgICBzZWxmLmRlY29tcG9zZSA9IGRlY29tcG9zZSAgICAgICAgICAgICAgICMgZ29hbF90ZXh0IC0+IFBsYW4KICAgICAgICBzZWxmLnJvdXRlciA9IHJvdXRlcgogICAgICAgIHNlbGYudmFsaWRhdGUgPSB2YWxpZGF0ZSBvciAobGFtYmRhIHN0ZXAsIG91dDogb3V0LmdldCgiYW5zd2VyIikgaXMgbm90IE5vbmUpCgogICAgZGVmIHBsYW4oc2VsZiwgZ29hbF90ZXh0OiBzdHIpIC0+IFBsYW46CiAgICAgICAgcmV0dXJuIHNlbGYuZGVjb21wb3NlKGdvYWxfdGV4dCkKCiAgICBkZWYgZXhlY3V0ZShzZWxmLCBwbGFuOiBQbGFuKSAtPiBkaWN0OgogICAgICAgIG9yZGVyID0gdG9wb19vcmRlcihwbGFuLnN0ZXBzKSAgICAgICAgICAgIyByYWlzZXMgb24gYSBiYWQgcGxhbgogICAgICAgIGJ5X2lkID0gcGxhbi5ieV9pZCgpCiAgICAgICAgcmVzdWx0cywgcmVwbGFubmVkID0ge30sIFtdCiAgICAgICAgZm9yIHNpZCBpbiBvcmRlcjoKICAgICAgICAgICAgc3RlcCA9IGJ5X2lkW3NpZF0KICAgICAgICAgICAgdGFzayA9IGRpY3Qoc3RlcC5wYXlsb2FkKQogICAgICAgICAgICB0YXNrWyJ0ZXh0Il0gPSBzdGVwLnRleHQKICAgICAgICAgICAgIyBUSFJFQUQgZWFjaCBkZXBlbmRlbmN5J3MgYW5zd2VyIGludG8gdGhpcyB0YXNrJ3MgY29udGV4dC4KICAgICAgICAgICAgdGFza1siY29udGV4dCJdID0ge2Q6IHJlc3VsdHNbZF0uZ2V0KCJhbnN3ZXIiKSBmb3IgZCBpbiBzdGVwLmRlcHN9CiAgICAgICAgICAgIG91dCA9IHNlbGYucm91dGVyLmRpc3BhdGNoKHRhc2spCiAgICAgICAgICAgIGlmIG5vdCBzZWxmLnZhbGlkYXRlKHN0ZXAsIG91dCk6CiAgICAgICAgICAgICAgICAjIFJFUExBTjogd2hvZXZlciB0aGUgcm91dGVyIHBpY2tlZCBmYWlsZWQuIEVzY2FsYXRlIHRoaXMgb25lCiAgICAgICAgICAgICAgICAjIHN0ZXAgc3RyYWlnaHQgdG8gdGhlIGdlbmVyYWxpc3QgKGl0IGtlZXBzIHRoZSB0aHJlYWRlZCBjb250ZXh0KS4KICAgICAgICAgICAgICAgIGcgPSBzZWxmLnJvdXRlci5nZW5lcmFsaXN0CiAgICAgICAgICAgICAgICBtID0gZy5hY3QoTWVzc2FnZSgicGxhbm5lciIsIGcubmFtZSwgInRhc2siLCB0YXNrKSkKICAgICAgICAgICAgICAgIG91dCA9IHsiYW5zd2VyIjogbS5jb250ZW50LmdldCgiYW5zd2VyIiksCiAgICAgICAgICAgICAgICAgICAgICAgInRva2VucyI6IG91dC5nZXQoInRva2VucyIsIDApICsgbS5tZXRhLmdldCgidG9rZW5zIiwgMCksCiAgICAgICAgICAgICAgICAgICAgICAgInRyYWNlIjogb3V0LmdldCgidHJhY2UiLCBbXSkgKyBbZy5uYW1lXSwKICAgICAgICAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6ICJyZXBsYW4ifQogICAgICAgICAgICAgICAgcmVwbGFubmVkLmFwcGVuZChzaWQpCiAgICAgICAgICAgIHJlc3VsdHNbc2lkXSA9IG91dAogICAgICAgIHJldHVybiB7ImdvYWwiOiBwbGFuLmdvYWwsICJvcmRlciI6IG9yZGVyLCAicmVzdWx0cyI6IHJlc3VsdHMsCiAgICAgICAgICAgICAgICAicmVwbGFubmVkIjogcmVwbGFubmVkLAogICAgICAgICAgICAgICAgImFuc3dlcnMiOiB7azogdi5nZXQoImFuc3dlciIpIGZvciBrLCB2IGluIHJlc3VsdHMuaXRlbXMoKX0sCiAgICAgICAgICAgICAgICAidG9rZW5zIjogc3VtKHYuZ2V0KCJ0b2tlbnMiLCAwKSBmb3IgdiBpbiByZXN1bHRzLnZhbHVlcygpKX0K",
    "orchestra/reliability.py": "IyBvcmNoZXN0cmEvcmVsaWFiaWxpdHkucHkgIC0tICB0aGUgbGF5ZXIgdGhhdCBtYWtlcyBvcmNoZXN0cmF0aW9uIHRydXN0d29ydGh5LgojIEw4MCdzIFBsYW5uZXIgZGlkIGEgU0lOR0xFLVNIT1QgcmVwbGFuIChvbmUgcmV0cnkgb250byB0aGUgZ2VuZXJhbGlzdCkuIFRoYXQKIyBpcyBub3QgcmVsaWFiaWxpdHkuIFJlbGlhYmlsaXR5IGlzIGEgZGlzY2lwbGluZWQgbGF5ZXI6IGRpc3Rpbmd1aXNoIGZhaWx1cmVzCiMgd29ydGggcmV0cnlpbmcgZnJvbSBvbmVzIHRoYXQgYXJlbid0LCByZXRyeSB0aGUgdHJhbnNpZW50IG9uZXMgd2l0aCBCQUNLT0ZGCiMgc28geW91IG5laXRoZXIgZ2l2ZSB1cCB0b28gZWFybHkgbm9yIGhhbW1lciBhIHN0cnVnZ2xpbmcgZGVwZW5kZW5jeSwgYm91bmQKIyBldmVyeSBjYWxsIHdpdGggYSBUSU1FT1VUIHNvIGEgaHVuZyBzdGVwIGNhbid0IHN0YWxsIHRoZSBwbGFuLCB0cmlwIGEgQ0lSQ1VJVAojIEJSRUFLRVIgc28gYSBwZXJzaXN0ZW50bHktYnJva2VuIGRlcGVuZGVuY3kgaXMgaXNvbGF0ZWQgaW5zdGVhZCBvZiByZXRyaWVkCiMgaW50byB0aGUgZ3JvdW5kLCBhbmQgZ2l2ZSBhIHBsYW4gYW4gZXhwbGljaXQgUEFSVElBTC1GQUlMVVJFIHBvbGljeSBzbwojICIyIG9mIDUgc3RlcHMgZmFpbGVkIiByZXNvbHZlcyB0byBhIGRlZmluZWQgZGVncmFkZWQgcmVzdWx0LCBub3QgYSBjcmFzaC4KaW1wb3J0IHRpbWUsIHJhbmRvbQppbXBvcnQgY29uY3VycmVudC5mdXR1cmVzIGFzIF9jZgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgT3B0aW9uYWwsIExpc3QsIERpY3QsIEFueQoKIyAtLS0tIEZhaWx1cmUgdGF4b25vbXk6IHRoZSBzaW5nbGUgbW9zdCBpbXBvcnRhbnQgZGlzdGluY3Rpb24gLS0tLS0tLS0tLS0tLS0tLS0KY2xhc3MgVHJhbnNpZW50KEV4Y2VwdGlvbik6CiAgICAiIiJXb3J0aCByZXRyeWluZzogdGltZW91dCwgcmF0ZS1saW1pdCwgNXh4LCBjb25uZWN0aW9uIHJlc2V0LCBibGlwLiIiIgpjbGFzcyBQZXJtYW5lbnQoRXhjZXB0aW9uKToKICAgICIiIk5PVCB3b3J0aCByZXRyeWluZzogYmFkIGlucHV0LCBhdXRoLCA0eHgsIGEgdmFsaWRhdGlvbiBlcnJvci4iIiIKY2xhc3MgVGltZW91dChUcmFuc2llbnQpOgogICAgIiIiQSBjYWxsIHRoYXQgb3ZlcnJhbiBpdHMgZGVhZGxpbmUuIFRyYW5zaWVudCBieSBuYXR1cmUuIiIiCmNsYXNzIENpcmN1aXRPcGVuKFRyYW5zaWVudCk6CiAgICAiIiJUaGUgYnJlYWtlciBpcyBvcGVuOyB0aGUgY2FsbCB3YXMgcmVqZWN0ZWQgd2l0aG91dCB0b3VjaGluZyB0aGUgYmFja2VuZC4iIiIKCiMgLS0tLSBUaW1lb3V0OiBib3VuZCB0aGUgQ0FMTEVSIGV2ZW4gaWYgdGhlIGNhbGxlZSB3b24ndCBjb29wZXJhdGUgLS0tLS0tLS0tLS0tCmRlZiBjYWxsX3dpdGhfdGltZW91dChmbiwgdGltZW91dF9zLCAqYSwgKiprKToKICAgICMgUHVyZSBQeXRob24gY2FuJ3Qgc2FmZWx5IGtpbGwgYSB0aHJlYWQsIHNvIHdlIHJ1biBmbiBpbiBhIHdvcmtlciBhbmQKICAgICMgYWJhbmRvbiBpdHMgcmVzdWx0IGlmIGl0IG92ZXJydW5zLiBUaGlzIGJvdW5kcyB0aGUgY2FsbGVyJ3MgbGF0ZW5jeSwKICAgICMgd2hpY2ggaXMgd2hhdCBhIGRlYWRsaW5lIGlzIGZvci4KICAgIHdpdGggX2NmLlRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz0xKSBhcyBleDoKICAgICAgICBmdXQgPSBleC5zdWJtaXQoZm4sICphLCAqKmspCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gZnV0LnJlc3VsdCh0aW1lb3V0PXRpbWVvdXRfcykKICAgICAgICBleGNlcHQgX2NmLlRpbWVvdXRFcnJvcjoKICAgICAgICAgICAgcmFpc2UgVGltZW91dCgiY2FsbCBleGNlZWRlZCAlLjNmcyIgJSB0aW1lb3V0X3MpCgojIC0tLS0gUmV0cnkgd2l0aCBleHBvbmVudGlhbCBiYWNrb2ZmICsgaml0dGVyIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpAZGF0YWNsYXNzCmNsYXNzIFJldHJ5UG9saWN5OgogICAgbWF4X2F0dGVtcHRzOiBpbnQgPSAzCiAgICBiYXNlX2RlbGF5OiBmbG9hdCA9IDAuMDUKICAgIGZhY3RvcjogZmxvYXQgPSAyLjAKICAgIG1heF9kZWxheTogZmxvYXQgPSAyLjAKICAgIGppdHRlcjogZmxvYXQgPSAwLjUgICAgICAgICAgICAjIGZyYWN0aW9uIG9mIGVhY2ggZGVsYXkgdGhhdCBpcyByYW5kb21pemVkCiAgICBkZWYgZGVsYXkoc2VsZiwgYXR0ZW1wdDogaW50LCBybmc9cmFuZG9tKSAtPiBmbG9hdDoKICAgICAgICAjIGF0dGVtcHQgaXMgMS1iYXNlZDogdGhlIHdhaXQgQUZURVIgYXR0ZW1wdCBrLCBiZWZvcmUgYXR0ZW1wdCBrKzEuCiAgICAgICAgcmF3ID0gbWluKHNlbGYubWF4X2RlbGF5LCBzZWxmLmJhc2VfZGVsYXkgKiAoc2VsZi5mYWN0b3IgKiogKGF0dGVtcHQgLSAxKSkpCiAgICAgICAgIyAiZnVsbCBqaXR0ZXIiIGluIFtyYXcqKDEtaml0dGVyKSwgcmF3XSBzcHJlYWRzIHJldHJpZXMgc28gTiBjbGllbnRzCiAgICAgICAgIyB0aGF0IGZhaWxlZCB0b2dldGhlciBkb24ndCBhbGwgcmV0cnkgaW4gbG9ja3N0ZXAgKHRodW5kZXJpbmcgaGVyZCkuCiAgICAgICAgcmV0dXJuIHJhdyAqICgxIC0gc2VsZi5qaXR0ZXIgKiBybmcucmFuZG9tKCkpCgpAZGF0YWNsYXNzCmNsYXNzIEF0dGVtcHQ6CiAgICBuOiBpbnQKICAgIG9rOiBib29sCiAgICBlcnJvcjogT3B0aW9uYWxbc3RyXQogICAgc2xlcHQ6IGZsb2F0CgpkZWYgcmV0cnkoZm4sIHBvbGljeTogUmV0cnlQb2xpY3ksIHNsZWVwPXRpbWUuc2xlZXAsIHJuZz1yYW5kb20pIC0+IGRpY3Q6CiAgICAjIFJldHJ5IE9OTFkgVHJhbnNpZW50IGZhaWx1cmVzLiBBIFBlcm1hbmVudCBmYWlsdXJlIGZhaWxzIEZBU1QgKG5vIHJldHJ5LAogICAgIyBubyBiYWNrb2ZmKSAtLSByZXRyeWluZyBhIGJhZCBpbnB1dCBqdXN0IHdhc3RlcyB0aW1lIGFuZCBtb25leS4KICAgIGF0dGVtcHRzOiBMaXN0W0F0dGVtcHRdID0gW10KICAgIGxhc3QgPSBOb25lCiAgICBmb3IgbiBpbiByYW5nZSgxLCBwb2xpY3kubWF4X2F0dGVtcHRzICsgMSk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB2YWwgPSBmbigpCiAgICAgICAgICAgIGF0dGVtcHRzLmFwcGVuZChBdHRlbXB0KG4sIFRydWUsIE5vbmUsIDAuMCkpCiAgICAgICAgICAgIHJldHVybiB7Im9rIjogVHJ1ZSwgInZhbHVlIjogdmFsLCAiYXR0ZW1wdHMiOiBhdHRlbXB0cywgInJlYXNvbiI6ICJvayJ9CiAgICAgICAgZXhjZXB0IFBlcm1hbmVudCBhcyBlOgogICAgICAgICAgICBhdHRlbXB0cy5hcHBlbmQoQXR0ZW1wdChuLCBGYWxzZSwgInBlcm1hbmVudDolcyIgJSBlLCAwLjApKQogICAgICAgICAgICByZXR1cm4geyJvayI6IEZhbHNlLCAidmFsdWUiOiBOb25lLCAiYXR0ZW1wdHMiOiBhdHRlbXB0cywgInJlYXNvbiI6ICJwZXJtYW5lbnQifQogICAgICAgIGV4Y2VwdCBUcmFuc2llbnQgYXMgZToKICAgICAgICAgICAgbGFzdCA9IHN0cihlKQogICAgICAgICAgICBpZiBuID09IHBvbGljeS5tYXhfYXR0ZW1wdHM6CiAgICAgICAgICAgICAgICBhdHRlbXB0cy5hcHBlbmQoQXR0ZW1wdChuLCBGYWxzZSwgInRyYW5zaWVudDolcyIgJSBlLCAwLjApKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZCA9IHBvbGljeS5kZWxheShuLCBybmcpCiAgICAgICAgICAgIGF0dGVtcHRzLmFwcGVuZChBdHRlbXB0KG4sIEZhbHNlLCAidHJhbnNpZW50OiVzIiAlIGUsIGQpKQogICAgICAgICAgICBzbGVlcChkKQogICAgcmV0dXJuIHsib2siOiBGYWxzZSwgInZhbHVlIjogTm9uZSwgImF0dGVtcHRzIjogYXR0ZW1wdHMsICJyZWFzb24iOiAiZXhoYXVzdGVkIiwgImxhc3QiOiBsYXN0fQoKIyAtLS0tIENpcmN1aXQgYnJlYWtlcjogc3RvcCBoYW1tZXJpbmcgYSBkZXBlbmRlbmN5IHRoYXQgaXMgY2xlYXJseSBkb3duIC0tLS0tLS0tCkBkYXRhY2xhc3MKY2xhc3MgQ2lyY3VpdEJyZWFrZXI6CiAgICBmYWlsX3RocmVzaG9sZDogaW50ID0gMyAgICAgICAjIGNvbnNlY3V0aXZlIGZhaWx1cmVzIC0+IE9QRU4KICAgIHJlc2V0X3RpbWVvdXQ6IGZsb2F0ID0gMC4yICAgICMgc2Vjb25kcyB0byBzdGF5IE9QRU4gYmVmb3JlIGEgSEFMRl9PUEVOIHByb2JlCiAgICBoYWxmX29wZW5fc3VjY2Vzc2VzOiBpbnQgPSAxICAjIHN1Y2Nlc3NlcyBpbiBIQUxGX09QRU4gbmVlZGVkIHRvIENMT1NFCiAgICBub3c6IENhbGxhYmxlID0gdGltZS5tb25vdG9uaWMKICAgIHN0YXRlOiBzdHIgPSAiY2xvc2VkIgogICAgZmFpbHM6IGludCA9IDAKICAgIG9wZW5lZF9hdDogZmxvYXQgPSAwLjAKICAgIGhhbGZfb2s6IGludCA9IDAKICAgIGRlZiBhbGxvdyhzZWxmKSAtPiBib29sOgogICAgICAgICMgT1BFTiByZWplY3RzIGluc3RhbnRseSAobm8gYmFja2VuZCBjYWxsKS4gQWZ0ZXIgdGhlIGNvb2xkb3duIHdlIGFsbG93CiAgICAgICAgIyBPTkUgcHJvYmUgKEhBTEZfT1BFTikgdG8gdGVzdCB3aGV0aGVyIHRoZSBkZXBlbmRlbmN5IGhhcyByZWNvdmVyZWQuCiAgICAgICAgaWYgc2VsZi5zdGF0ZSA9PSAib3BlbiI6CiAgICAgICAgICAgIGlmIHNlbGYubm93KCkgLSBzZWxmLm9wZW5lZF9hdCA+PSBzZWxmLnJlc2V0X3RpbWVvdXQ6CiAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLCBzZWxmLmhhbGZfb2sgPSAiaGFsZl9vcGVuIiwgMAogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGRlZiBvbl9zdWNjZXNzKHNlbGYpOgogICAgICAgIGlmIHNlbGYuc3RhdGUgPT0gImhhbGZfb3BlbiI6CiAgICAgICAgICAgIHNlbGYuaGFsZl9vayArPSAxCiAgICAgICAgICAgIGlmIHNlbGYuaGFsZl9vayA+PSBzZWxmLmhhbGZfb3Blbl9zdWNjZXNzZXM6CiAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLCBzZWxmLmZhaWxzID0gImNsb3NlZCIsIDAKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmZhaWxzID0gMAogICAgZGVmIG9uX2ZhaWx1cmUoc2VsZik6CiAgICAgICAgaWYgc2VsZi5zdGF0ZSA9PSAiaGFsZl9vcGVuIjoKICAgICAgICAgICAgc2VsZi5zdGF0ZSwgc2VsZi5vcGVuZWRfYXQgPSAib3BlbiIsIHNlbGYubm93KCkgICAjIHByb2JlIGZhaWxlZCAtPiByZW9wZW4KICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmZhaWxzICs9IDEKICAgICAgICAgICAgaWYgc2VsZi5mYWlscyA+PSBzZWxmLmZhaWxfdGhyZXNob2xkOgogICAgICAgICAgICAgICAgc2VsZi5zdGF0ZSwgc2VsZi5vcGVuZWRfYXQgPSAib3BlbiIsIHNlbGYubm93KCkKCiMgLS0tLSBSZWxpYWJsZTogY29tcG9zZSBicmVha2VyIChvdXRlcikgKyB0aW1lb3V0ICsgcmV0cnkgKGlubmVyKSAtLS0tLS0tLS0tLS0tCmNsYXNzIFJlbGlhYmxlOgogICAgIyBPbmUgY2FsbCgpID0gYnJlYWtlciBnYXRlIC0+ICh0aW1lb3V0LXdyYXBwZWQpIHJldHJ5IC0+IHJlcG9ydCB0aGUgRklOQUwKICAgICMgb3V0Y29tZSB0byB0aGUgYnJlYWtlci4gQnJlYWtlciBpcyB0aGUgT1VURVIgZ3VhcmQgc28gYW4gb3BlbiBjaXJjdWl0CiAgICAjIHJlamVjdHMgZmFzdCBXSVRIT1VUIHNwZW5kaW5nIHRoZSByZXRyeSBidWRnZXQgb24gYmFja29mZiBzbGVlcHMuCiAgICBkZWYgX19pbml0X18oc2VsZiwgcG9saWN5OiBSZXRyeVBvbGljeSwgYnJlYWtlcjogT3B0aW9uYWxbQ2lyY3VpdEJyZWFrZXJdID0gTm9uZSwKICAgICAgICAgICAgICAgICB0aW1lb3V0X3M6IE9wdGlvbmFsW2Zsb2F0XSA9IE5vbmUpOgogICAgICAgIHNlbGYucG9saWN5LCBzZWxmLmJyZWFrZXIsIHNlbGYudGltZW91dF9zID0gcG9saWN5LCBicmVha2VyLCB0aW1lb3V0X3MKICAgICAgICBzZWxmLnJlamVjdGVkID0gMAogICAgZGVmIGNhbGwoc2VsZiwgZm4sIHNsZWVwPXRpbWUuc2xlZXAsIHJuZz1yYW5kb20pIC0+IGRpY3Q6CiAgICAgICAgaWYgc2VsZi5icmVha2VyIGFuZCBub3Qgc2VsZi5icmVha2VyLmFsbG93KCk6CiAgICAgICAgICAgIHNlbGYucmVqZWN0ZWQgKz0gMQogICAgICAgICAgICByZXR1cm4geyJvayI6IEZhbHNlLCAidmFsdWUiOiBOb25lLCAiYXR0ZW1wdHMiOiBbXSwgInJlYXNvbiI6ICJjaXJjdWl0X29wZW4ifQogICAgICAgIHRhcmdldCA9IGZuIGlmIHNlbGYudGltZW91dF9zIGlzIE5vbmUgZWxzZSAobGFtYmRhOiBjYWxsX3dpdGhfdGltZW91dChmbiwgc2VsZi50aW1lb3V0X3MpKQogICAgICAgIHJlcyA9IHJldHJ5KHRhcmdldCwgc2VsZi5wb2xpY3ksIHNsZWVwPXNsZWVwLCBybmc9cm5nKQogICAgICAgIGlmIHNlbGYuYnJlYWtlcjoKICAgICAgICAgICAgc2VsZi5icmVha2VyLm9uX3N1Y2Nlc3MoKSBpZiByZXNbIm9rIl0gZWxzZSBzZWxmLmJyZWFrZXIub25fZmFpbHVyZSgpCiAgICAgICAgcmV0dXJuIHJlcwoKIyAtLS0tIFBhcnRpYWwtZmFpbHVyZSBwb2xpY3kgZm9yIGEgcGFyYWxsZWwgbGF5ZXIgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIHJ1bl9sYXllcihzdGVwX2lkczogTGlzdFtzdHJdLCBydW5fb25lOiBDYWxsYWJsZVtbc3RyXSwgZGljdF0sCiAgICAgICAgICAgICAgcG9saWN5OiBzdHIgPSAiYWxsX29yX25vdGhpbmciLCBxdW9ydW06IE9wdGlvbmFsW2ludF0gPSBOb25lKSAtPiBkaWN0OgogICAgIyBydW5fb25lKHNpZCkgLT4geyJvayI6IGJvb2wsIC4uLn0uIERlY2lkZSBhIExBWUVSIHN0YXR1cyB3aGVuIHNvbWUgc3RlcHMKICAgICMgZmFpbCwgaW5zdGVhZCBvZiBsZXR0aW5nIG9uZSBmYWlsdXJlIGJlY29tZSBhbiB1bmhhbmRsZWQgZXhjZXB0aW9uLgogICAgcmVzdWx0cyA9IHtzaWQ6IHJ1bl9vbmUoc2lkKSBmb3Igc2lkIGluIHN0ZXBfaWRzfQogICAgb2sgID0gW3MgZm9yIHMgaW4gc3RlcF9pZHMgaWYgcmVzdWx0c1tzXVsib2siXV0KICAgIGJhZCA9IFtzIGZvciBzIGluIHN0ZXBfaWRzIGlmIG5vdCByZXN1bHRzW3NdWyJvayJdXQogICAgbiA9IGxlbihzdGVwX2lkcykKICAgIGlmIHBvbGljeSA9PSAiYWxsX29yX25vdGhpbmciOgogICAgICAgIHN0YXR1cyA9ICJvayIgaWYgbm90IGJhZCBlbHNlICJmYWlsZWQiCiAgICBlbGlmIHBvbGljeSA9PSAiYmVzdF9lZmZvcnQiOgogICAgICAgIHN0YXR1cyA9ICJvayIgaWYgbm90IGJhZCBlbHNlICJkZWdyYWRlZCIKICAgIGVsaWYgcG9saWN5ID09ICJxdW9ydW0iOgogICAgICAgIG5lZWQgPSBxdW9ydW0gaWYgcXVvcnVtIGlzIG5vdCBOb25lIGVsc2UgKG4gLy8gMiArIDEpCiAgICAgICAgc3RhdHVzID0gIm9rIiBpZiBsZW4ob2spID49IG5lZWQgZWxzZSAiZmFpbGVkIgogICAgZWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmtub3duIHBhcnRpYWwtZmFpbHVyZSBwb2xpY3kgJXIiICUgcG9saWN5KQogICAgcmV0dXJuIHsic3RhdHVzIjogc3RhdHVzLCAib2siOiBvaywgImZhaWxlZCI6IGJhZCwgInJlc3VsdHMiOiByZXN1bHRzLAogICAgICAgICAgICAicG9saWN5IjogcG9saWN5LCAibiI6IG59Cg==",
    "orchestra/__init__.py": "IiIib3JjaGVzdHJhIOKAlCBhIHNtYWxsLCB0ZXN0ZWQgbXVsdGktYWdlbnQgb3JjaGVzdHJhdGlvbiB0b29sa2l0LgoKRml2ZSBjb21wb3NhYmxlIHBpbGxhcnMsIGVhY2ggYnVpbHQgaW4gb25lIGxlc3NvbiBvZiBQaGFzZSA5OgogICAgY29yZSAgICAgICAgIChMNzcpICBNZXNzYWdlICsgQWdlbnQgcHJpbWl0aXZlczsgdGhlIGNvbnRyb2wtZmxvdyBzdWJzdHJhdGUuCiAgICBibGFja2JvYXJkICAgKEw3OCkgIGEgdmVyc2lvbmVkLCBjb25jdXJyZW5jeS1zYWZlIHNoYXJlZCB3b3Jrc3BhY2UuCiAgICByb3V0ZXIgICAgICAgKEw3OSkgIGNsYXNzaWZ5IGEgdGFzaywgZGlzcGF0Y2ggdG8gYSBzcGVjaWFsaXN0LCBlc2NhbGF0ZS4KICAgIHBsYW5uZXIgICAgICAoTDgwKSAgZGVjb21wb3NlIGEgZ29hbCBpbnRvIGFuIG9yZGVyZWQgREFHIG9mIHN1Yi10YXNrcy4KICAgIHJlbGlhYmlsaXR5ICAoTDgxKSAgcmV0cmllcywgdGltZW91dHMsIGNpcmN1aXQgYnJlYWtlcnMsIHBhcnRpYWwtZmFpbHVyZS4KClRoZSBjYXBzdG9uZSAoTDgyKSBwcm92ZXMgdGhleSBDT01QT1NFOiBSZXNpbGllbnRPcmNoZXN0cmF0b3Igd2lyZXMgYWxsIGZpdmUKaW50byBvbmUgdGhpbmcgdGhhdCBkZWNvbXBvc2VzIGEgZ29hbCwgZGlzcGF0Y2hlcyBlYWNoIHN0ZXAgdG8gdGhlIHJpZ2h0CnNwZWNpYWxpc3QsIHdyYXBzIGV2ZXJ5IGNhbGwgaW4gdGhlIHJlbGlhYmlsaXR5IGxheWVyLCByZWNvcmRzIHJlc3VsdHMgb24gYQpibGFja2JvYXJkLCBhbmQgcmVzb2x2ZXMgZWFjaCBwYXJhbGxlbCBsYXllciB3aXRoIGEgcGFydGlhbC1mYWlsdXJlIHBvbGljeS4KIiIiCl9fdmVyc2lvbl9fID0gIjAuMS4wIgoKZnJvbSBvcmNoZXN0cmEuY29yZSBpbXBvcnQgTWVzc2FnZSwgQWdlbnQKZnJvbSBvcmNoZXN0cmEuYmxhY2tib2FyZCBpbXBvcnQgQmxhY2tib2FyZCwgS25vd2xlZGdlU291cmNlLCBydW5fdW50aWxfcXVpZXNjZW50CmZyb20gb3JjaGVzdHJhLnJvdXRlciBpbXBvcnQgUm91dGUsIENsYXNzaWZpZXIsIFJvdXRlcgpmcm9tIG9yY2hlc3RyYS5wbGFubmVyIGltcG9ydCBTdGVwLCBQbGFuLCB0b3BvX29yZGVyLCBsYXllcnMsIFBsYW5uZXIKZnJvbSBvcmNoZXN0cmEucmVsaWFiaWxpdHkgaW1wb3J0ICgKICAgIFRyYW5zaWVudCwgUGVybWFuZW50LCBUaW1lb3V0LCBDaXJjdWl0T3BlbiwKICAgIGNhbGxfd2l0aF90aW1lb3V0LCBSZXRyeVBvbGljeSwgQXR0ZW1wdCwgcmV0cnksCiAgICBDaXJjdWl0QnJlYWtlciwgUmVsaWFibGUsIHJ1bl9sYXllciwKKQoKX19hbGxfXyA9IFsKICAgICJfX3ZlcnNpb25fXyIsCiAgICAjIGNvcmUKICAgICJNZXNzYWdlIiwgIkFnZW50IiwKICAgICMgYmxhY2tib2FyZAogICAgIkJsYWNrYm9hcmQiLCAiS25vd2xlZGdlU291cmNlIiwgInJ1bl91bnRpbF9xdWllc2NlbnQiLAogICAgIyByb3V0ZXIKICAgICJSb3V0ZSIsICJDbGFzc2lmaWVyIiwgIlJvdXRlciIsCiAgICAjIHBsYW5uZXIKICAgICJTdGVwIiwgIlBsYW4iLCAidG9wb19vcmRlciIsICJsYXllcnMiLCAiUGxhbm5lciIsCiAgICAjIHJlbGlhYmlsaXR5CiAgICAiVHJhbnNpZW50IiwgIlBlcm1hbmVudCIsICJUaW1lb3V0IiwgIkNpcmN1aXRPcGVuIiwKICAgICJjYWxsX3dpdGhfdGltZW91dCIsICJSZXRyeVBvbGljeSIsICJBdHRlbXB0IiwgInJldHJ5IiwKICAgICJDaXJjdWl0QnJlYWtlciIsICJSZWxpYWJsZSIsICJydW5fbGF5ZXIiLApdCg==",
    "pyproject.toml": "W2J1aWxkLXN5c3RlbV0KcmVxdWlyZXMgPSBbImhhdGNobGluZyJdCmJ1aWxkLWJhY2tlbmQgPSAiaGF0Y2hsaW5nLmJ1aWxkIgoKW3Byb2plY3RdCm5hbWUgPSAib3JjaGVzdHJhLWFnZW50cyIKdmVyc2lvbiA9ICIwLjEuMCIKZGVzY3JpcHRpb24gPSAiQSBzbWFsbCwgdGVzdGVkIG11bHRpLWFnZW50IG9yY2hlc3RyYXRpb24gdG9vbGtpdDogcm91dGluZywgcGxhbm5pbmcsIHNoYXJlZCBzdGF0ZSwgYW5kIHJlbGlhYmlsaXR5LiIKcmVhZG1lID0gIlJFQURNRS5tZCIKcmVxdWlyZXMtcHl0aG9uID0gIj49My4xMCIKbGljZW5zZSA9IHsgdGV4dCA9ICJNSVQiIH0KYXV0aG9ycyA9IFt7IG5hbWUgPSAiR291cmF2IEtoYW5pam9lIiB9XQprZXl3b3JkcyA9IFsibGxtIiwgImFnZW50cyIsICJvcmNoZXN0cmF0aW9uIiwgIm11bHRpLWFnZW50IiwgInJlbGlhYmlsaXR5Il0KZGVwZW5kZW5jaWVzID0gW10KCltwcm9qZWN0LnVybHNdCkhvbWVwYWdlID0gImh0dHBzOi8vZ2l0aHViLmNvbS9nb3VyYXYvb3JjaGVzdHJhLWFnZW50cyIKClt0b29sLmhhdGNoLmJ1aWxkLnRhcmdldHMud2hlZWxdCnBhY2thZ2VzID0gWyJvcmNoZXN0cmEiXQo=",
    "README.md": "IyBvcmNoZXN0cmEtYWdlbnRzCgpBIHNtYWxsLCAqKnRlc3RlZCoqIG11bHRpLWFnZW50IG9yY2hlc3RyYXRpb24gdG9vbGtpdC4gRml2ZSBjb21wb3NhYmxlIHBpbGxhcnM6Cgp8IGltcG9ydCB8IHBpbGxhciB8IHdoYXQgaXQgZG9lcyB8CnwtLS18LS0tfC0tLXwKfCBgb3JjaGVzdHJhLmNvcmVgIHwgcHJpbWl0aXZlcyB8IGBNZXNzYWdlYCArIGBBZ2VudGA6IHRoZSBjb250cm9sLWZsb3cgc3Vic3RyYXRlIHwKfCBgb3JjaGVzdHJhLmJsYWNrYm9hcmRgIHwgc2hhcmVkIHN0YXRlIHwgdmVyc2lvbmVkLCBjb25jdXJyZW5jeS1zYWZlIHdvcmtzcGFjZSB8CnwgYG9yY2hlc3RyYS5yb3V0ZXJgIHwgcm91dGluZyB8IGNsYXNzaWZ5IGEgdGFzaywgZGlzcGF0Y2ggdG8gYSBzcGVjaWFsaXN0LCBlc2NhbGF0ZSB8CnwgYG9yY2hlc3RyYS5wbGFubmVyYCB8IHBsYW5uaW5nIHwgZGVjb21wb3NlIGEgZ29hbCBpbnRvIGFuIG9yZGVyZWQgREFHIG9mIHN1Yi10YXNrcyB8CnwgYG9yY2hlc3RyYS5yZWxpYWJpbGl0eWAgfCByZWxpYWJpbGl0eSB8IHJldHJpZXMsIHRpbWVvdXRzLCBjaXJjdWl0IGJyZWFrZXJzLCBwYXJ0aWFsLWZhaWx1cmUgfAoKPiBEaXN0cmlidXRpb24gbmFtZSBpcyAqKmBvcmNoZXN0cmEtYWdlbnRzYCoqOyB5b3UgaW1wb3J0IGl0IGFzICoqYG9yY2hlc3RyYWAqKi4KClBhcnQgb2YgYSBmb3VyLXRvb2wgcG9ydGZvbGlvOiAqKnBhcGVyLWRpc3RpbGxlcioqIChhbiBhZ2VudCksICoqYWdlbnQtYmVuY2gqKgoob2ZmbGluZSBldmFsKSwgKiphZ2VudC1vYnMqKiAocHJvZHVjdGlvbiBvcHMpLCBhbmQgKipvcmNoZXN0cmEqKiAobXVsdGktYWdlbnQpLgo="
}
for relpath, b64 in _FILES.items():
    dst = os.path.join(PROJ, relpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "wb") as f:
        f.write(base64.b64decode(b64))

# Import the freshly-written package from the source tree.
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)
importlib.invalidate_caches()
import orchestra
importlib.reload(orchestra)

print("wrote project to:", PROJ)
for root, _, files in os.walk(PROJ):
    for fn in sorted(files):
        rel = os.path.relpath(os.path.join(root, fn), PROJ)
        print("   ", rel)
print()
print("import name : orchestra   version:", orchestra.__version__)
print("public API  :", len(orchestra.__all__), "symbols")


## §1 · Consolidation: one package, one public API

Five files in a folder is not yet a package a stranger can use. Three things
turn it into one:

1. **`orchestra/__init__.py`** re-exports the whole public surface and declares
   `__all__` (25 symbols) and `__version__`. Now `from orchestra import Router,
   Planner, Reliable` works — the user never needs to know which submodule a
   name lives in.
2. **`pyproject.toml`** describes how to *build* it (hatchling backend). Note the
   deliberate split: the **distribution name is `orchestra-agents`** (what you
   `pip install`, because `orchestra` was taken on PyPI) but the **import name is
   `orchestra`**. Distribution name ≠ import name is normal and worth knowing.
3. **`README.md`** — the first thing anyone reads. `pyproject` *references* it,
   so it has to exist before we build (a mistake worth remembering: any file
   `pyproject` points at must be written in the same or an earlier cell than the
   build step).

A pleasant surprise from consolidation: **the five modules share zero symbol
names.** (Contrast the `agent-obs` capstone, where two module pairs both defined
`redact` and `burn_rate` and the `__init__` had to arbitrate.) The next cell
proves that statically, before we trust a single import.

In [ ]:
import ast
CHECKS = []
def check(name, cond):
    CHECKS.append((name, bool(cond)))
    print(("PASS" if cond else "FAIL"), "-", name)

# The symbols each module PROMISES to expose (its contract with __init__).
PROMISED = {
    "core.py":        ["Message", "Agent"],
    "blackboard.py":  ["Blackboard", "KnowledgeSource", "run_until_quiescent"],
    "router.py":      ["Route", "Classifier", "Router"],
    "planner.py":     ["Step", "Plan", "topo_order", "layers", "Planner"],
    "reliability.py": ["Transient", "Permanent", "Timeout", "CircuitOpen",
                       "call_with_timeout", "RetryPolicy", "Attempt", "retry",
                       "CircuitBreaker", "Reliable", "run_layer"],
}

def top_level_defs(path):
    tree = ast.parse(open(path).read())
    names = set()
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            names.add(node.name)
    return names

pkg_dir = os.path.join(PROJ, "orchestra")
defs_by_mod = {}
for mod, promised in PROMISED.items():
    defs = top_level_defs(os.path.join(pkg_dir, mod))
    defs_by_mod[mod] = defs
    missing = [s for s in promised if s not in defs]
    check("%-15s exposes all %d promised symbols" % (mod, len(promised)), not missing)

# Every promised symbol must actually be importable from the top-level package.
importable = all(hasattr(orchestra, s) for ss in PROMISED.values() for s in ss)
check("every promised symbol is importable from `orchestra`", importable)

# __all__ must cover every promised symbol (nothing silently unexported).
covered = all(s in orchestra.__all__ for ss in PROMISED.values() for s in ss)
check("__all__ covers every promised symbol", covered)

# The happy surprise: zero symbol-name collisions across the five modules.
from collections import Counter
counts = Counter(s for defs in defs_by_mod.values() for s in defs)
collisions = [s for s, n in counts.items() if n > 1]
check("zero symbol-name collisions across the five modules", not collisions)
print("\ncollisions:", collisions or "(none)")

## §2 · The load-bearing cell: one run, all five pillars

Here is the whole reason a capstone exists. `ResilientOrchestrator` is a *new*
class — it lives in the lesson, not the package (the package ships primitives;
the orchestrator is one way to wire them). It uses **every pillar at once**:

- **planner** — `layers(plan.steps)` groups steps into dependency-ordered,
  internally-parallel layers.
- **router** — each step is dispatched to a specialist (or escalated).
- **reliability** — every dispatch runs through `Reliable.call` (retry / timeout
  / breaker), and every layer is resolved by `run_layer` with a partial-failure
  policy.
- **blackboard** — results are written to a versioned `Blackboard`, and each
  step reads its dependencies' answers back *off the board* (not from a local
  dict) — that's what threads a computed total into the billing step.
- **core** — `Message`/`Agent` are the substrate all of the above ride on.

The scenario: *"compute the total, then bill the customer."* Billing is a
**flaky** service — it 503s on the first call for each step, then works. A naive
L80 planner would crash on that blip. The orchestrator's reliability layer
absorbs it, and the answer flows math → blackboard → billing.

In [ ]:
import re
from orchestra import (Message, Agent, Blackboard, Classifier, Router,
                       Step, Plan, layers, RetryPolicy, Reliable, run_layer, Transient)

# ── The integration piece: ResilientOrchestrator (ties all five together) ────
class ResilientOrchestrator:
    """planner.layers -> for each layer, run_layer(...) where each step is
    dispatched through the router inside Reliable.call, with results/versions
    living on a Blackboard and dependencies threaded by reading the board."""
    def __init__(self, decompose, router, reliable_factory, validate=None):
        self.decompose = decompose
        self.router = router
        self.reliable_factory = reliable_factory          # () -> a fresh Reliable
        self.validate = validate or (lambda step, out: out.get("answer") is not None)

    def plan(self, goal):
        return self.decompose(goal)

    def execute(self, plan, policy="all_or_nothing"):
        bb = Blackboard()                                 # PILLAR: shared state
        by_id = plan.by_id()
        replanned, failed, layer_status = [], [], {}
        for layer in layers(plan.steps):                  # PILLAR: planner
            def run_one(sid):
                step = by_id[sid]
                task = dict(step.payload); task["text"] = step.text
                # thread deps by READING their answers off the board
                task["context"] = {d: (bb.get(d) or {}).get("answer") for d in step.deps}
                rel = self.reliable_factory()             # PILLAR: reliability
                r = rel.call(lambda: self.router.dispatch(task))   # PILLAR: router
                if r["ok"] and self.validate(step, r["value"]):
                    bb.set(sid, r["value"]); return {"ok": True, "sid": sid}
                # retries couldn't save it -> replan onto the generalist
                g = self.router.generalist
                m = g.act(Message("orch", g.name, "task", task))  # PILLAR: core
                out = {"answer": m.content.get("answer"), "reason": "replan"}
                bb.set(sid, out)
                ok = self.validate(step, out)
                (replanned if ok else failed).append(sid)
                return {"ok": ok, "sid": sid}
            layer_status[tuple(layer)] = run_layer(layer, run_one, policy=policy)["status"]
        return {"goal": plan.goal, "bb": bb, "replanned": replanned, "failed": failed,
                "layer_status": layer_status,
                "answers": {sid: (bb.get(sid) or {}).get("answer") for sid in by_id}}

# ── The deterministic world (faithful to L79-L81) ────────────────────────────
def math_backend(name, task):
    t = task["text"].lower()
    st = re.search(r"subtotal (\d+(?:\.\d+)?)", t); tx = re.search(r"tax (\d+(?:\.\d+)?)", t)
    if st and tx:
        return ({"answer": {"total": round(float(st.group(1)) * (1 + float(tx.group(1))), 2)}}, 20)
    return ({"answer": None}, 20)

class FlakyBilling:
    """503 on the FIRST call for a given step, then succeeds. A real blip."""
    def __init__(self): self.seen = {}
    def backend(self, name, task):
        k = task["text"]; self.seen[k] = self.seen.get(k, 0) + 1
        if self.seen[k] == 1:
            raise Transient("billing service 503")
        amount = None
        for v in (task.get("context") or {}).values():
            if isinstance(v, dict) and "total" in v: amount = v["total"]
        if amount is None: return ({"answer": None}, 25)
        return ({"answer": {"invoice": "Billed $%.2f" % amount, "amount": amount}}, 25)

def gen_backend(name, task):
    if "bill" in task["text"].lower():                    # an honest generalist:
        return ({"answer": None}, 60)                     # it can't invent a bill
    return ({"answer": {"note": "generalist fallback"}}, 60)

flaky = FlakyBilling()
math_agent = Agent("math-bot", "math", math_backend)
bill_agent = Agent("bill-bot", "billing", flaky.backend)
generalist = Agent("generalist", "general", gen_backend)
clf = Classifier({"math": ["compute", "total", "subtotal", "tax"],
                  "billing": ["bill", "invoice", "customer"]})
router = Router(clf, {"math": math_agent, "billing": bill_agent}, generalist, threshold=0.6)

def decompose(goal):
    return Plan(goal, [
        Step("s0", "compute total subtotal 100 tax 0.1", [], {"category": "math"}),
        Step("s1", "bill customer acme for the invoice", ["s0"], {"category": "billing"}),
    ])

# ═══ RUN IT: one execute(), all five pillars ═════════════════════════════════
flaky.seen.clear()
orch = ResilientOrchestrator(decompose, router,
    reliable_factory=lambda: Reliable(RetryPolicy(max_attempts=3, base_delay=0.001, jitter=0.0)))
out = orch.execute(orch.plan("compute the total then bill the customer"))
bb = out["bb"]

print("answers   :", out["answers"])
print("bb keys   :", sorted(bb.keys()), "| versions:", {k: bb.version(k) for k in sorted(bb.keys())})
print("layers    :", layers(decompose("x").steps), "| status:", list(out["layer_status"].values()))

check("planner ordered the plan into layers [[s0],[s1]]",
      layers(decompose("x").steps) == [["s0"], ["s1"]])
check("router+reliability completed the FLAKY billing step",
      out["answers"]["s1"] is not None)
check("dependency threaded THROUGH the blackboard: billed amount == total == 110.0",
      out["answers"]["s1"]["amount"] == 110.0)
check("reliability retry absorbed the blip -> no replan needed",
      out["replanned"] == [] and out["failed"] == [])
check("blackboard recorded every step with a version bump",
      bb.version("s0") == 1 and bb.version("s1") == 1)
check("all layers resolved ok under all_or_nothing",
      set(out["layer_status"].values()) == {"ok"})

## §3 · When a dependency is *permanently* down

Retries help a blip; they can't resurrect a service that's genuinely gone. The
capstone has to show the system **degrading gracefully instead of crashing or
lying**. Same orchestrator, but now billing is hard-down *and* the generalist
is honest — it won't fabricate an invoice it can't produce. Under a
`best_effort` policy the billing layer resolves as **`degraded`**, the math
answer is preserved, and `execute()` still returns a structured result. Nothing
raises.

In [ ]:
class DeadBilling:
    def backend(self, name, task):
        raise Transient("billing hard-down")   # never recovers

dead_router = Router(clf,
    {"math": math_agent, "billing": Agent("bill-bot", "billing", DeadBilling().backend)},
    generalist, threshold=0.6)

orch2 = ResilientOrchestrator(decompose, dead_router,
    reliable_factory=lambda: Reliable(RetryPolicy(max_attempts=2, base_delay=0.001, jitter=0.0)))
out2 = orch2.execute(orch2.plan("compute the total then bill the customer"),
                     policy="best_effort")

print("answers     :", out2["answers"])
print("failed steps:", out2["failed"], "| layer status:", list(out2["layer_status"].values()))

check("a permanently-dead step (retry can't help, generalist can't rescue) is marked FAILED",
      "s1" in out2["failed"])
check("best_effort PRESERVES the good math answer (110.0)",
      out2["answers"]["s0"]["total"] == 110.0)
check("its layer resolves 'degraded' -> not a crash, not a fabricated answer",
      "degraded" in out2["layer_status"].values())
check("execute() returned a structured result (nothing was raised)",
      isinstance(out2["answers"], dict))

## §4 · Build the distribution

An integration test proves the *code* composes. A wheel proves the *artifact*
installs. We build both distribution formats — an **sdist** (`.tar.gz`, the
source) and a **wheel** (`.whl`, the pre-built package pip prefers) — then run
`twine check`, which validates the metadata a package index will reject you for
getting wrong (bad README rendering, missing fields, etc.).

> On Colab this cell installs `build` + `twine` from PyPI, so it needs network
> the first time. It builds *from* `PROJ`, where we wrote `pyproject.toml`.

In [ ]:
# Build the sdist + wheel, then validate metadata with twine.
def run(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr)

# Ensure the build toolchain is present (quiet; harmless if already installed).
run([sys.executable, "-m", "pip", "install", "-q", "build", "twine"])

rc, log = run([sys.executable, "-m", "build"], cwd=PROJ)
print(log.strip().splitlines()[-1] if log.strip() else "(no output)")

dist = os.path.join(PROJ, "dist")
artifacts = sorted(os.listdir(dist)) if os.path.isdir(dist) else []
print("dist/:", artifacts)

has_wheel = any(a.endswith(".whl") for a in artifacts)
has_sdist = any(a.endswith(".tar.gz") for a in artifacts)
check("`python -m build` produced a wheel (.whl)", has_wheel)
check("`python -m build` produced an sdist (.tar.gz)", has_sdist)

rc2, log2 = run([sys.executable, "-m", "twine", "check", os.path.join(dist, "*")])
# twine needs a real glob; expand it ourselves for portability.
import glob
rc2, log2 = run([sys.executable, "-m", "twine", "check"] + glob.glob(os.path.join(dist, "*")))
print(log2.strip())
check("twine check PASSED for every built artifact",
      rc2 == 0 and "PASSED" in log2 and "FAILED" not in log2)

## §5 · The proof that matters: a *fresh* wheel in a *clean* process

Everything above imported `orchestra` from the source tree — the code we've had
on `sys.path` the whole notebook. That does **not** prove the built wheel works;
it only proves the source works. The honest test is to install the freshly-built
`.whl` into an isolated directory and import it from a **brand-new subprocess**
whose only path to the package is that directory. If that subprocess can
`import orchestra` and read its version, a stranger's `pip install` will too.

In [ ]:
import glob, shutil

wheel = glob.glob(os.path.join(PROJ, "dist", "*.whl"))[0]
target = os.path.join(BASE, "_fresh_install")
shutil.rmtree(target, ignore_errors=True)
run([sys.executable, "-m", "pip", "install", "-q", wheel, "--target", target])

# (a) NEGATIVE control: without the target dir, orchestra must NOT import.
neg = subprocess.run(
    [sys.executable, "-c", "import orchestra"],
    capture_output=True, text=True, cwd=BASE)
check("orchestra is NOT importable without the wheel dir (clean baseline)",
      neg.returncode != 0)

# (b) POSITIVE: prepend ONLY the target dir; orchestra must resolve from there.
probe = (
    "import sys; sys.path.insert(0, %r);"
    "import orchestra;"
    "from orchestra import Router, Planner, Reliable, Blackboard, run_layer;"
    "print('FRESH-WHEEL OK', orchestra.__version__, len(orchestra.__all__),"
    " orchestra.__file__)" % target)
pos = subprocess.run([sys.executable, "-c", probe],
                     capture_output=True, text=True, cwd=BASE)
print(pos.stdout.strip() or pos.stderr.strip())
check("the freshly-built wheel imports in a clean subprocess",
      pos.returncode == 0 and "FRESH-WHEEL OK" in pos.stdout)
check("the imported package came from the wheel install dir",
      target in pos.stdout)

## §6 · Launch day — your fourth open-source artifact

You now have four tools that *actually reference each other*:

- **paper-distiller** — an agent that does a real job.
- **agent-bench** — offline evaluation for agents.
- **agent-obs** — production observability for agents.
- **orchestra** — multi-agent orchestration (this one).

Portfolio coherence is the point. `orchestra` is the natural thing to *observe
with* `agent-obs` and *evaluate with* `agent-bench`, and a multi-agent system is
exactly where you'd deploy the `paper-distiller` agent as one specialist. The
READMEs should cross-promote **because the code genuinely connects**, not as
empty link-swapping. Stagger the launch ~1 week from the others so it reads as a
deliberate series, and don't reuse the same "awesome-list" you already submitted
to.

The next cell writes the launch scaffolding: a launch checklist, a curated
good-first-issue, and CI/release workflows.

In [ ]:
LAUNCH_CHECKLIST = """# orchestra-agents — launch checklist

## Before you tag v0.1.0
- [ ] `python -m build` produces a wheel + sdist locally
- [ ] `twine check dist/*` PASSES
- [ ] Fresh-wheel import works in a clean venv (`pip install dist/*.whl`)
- [ ] README renders on GitHub (headings, table, code block)
- [ ] LICENSE present (MIT) and named in pyproject
- [ ] The integration test (all five pillars) is in the test suite, not just the notebook

## Portfolio coherence (the point of a 4th repo)
- [ ] README cross-links paper-distiller / agent-bench / agent-obs
- [ ] One concrete example wires orchestra + agent-obs (a span per step)
- [ ] Launch ~1 week apart from the sibling repos
- [ ] Submit to a DIFFERENT awesome-list than the earlier tools

## Launch day
- [ ] Tag `v0.1.0`, let the release workflow publish
- [ ] Short write-up: the problem (composing agents reliably), the five pillars,
      the one integration test
- [ ] Pin the good-first-issue so a first contributor has an on-ramp
"""

FIRST_ISSUE = """# good first issue: add a `Reliable`-aware timeout to `ResilientOrchestrator`

`Reliable` already supports `timeout_s`, but the orchestrator's
`reliable_factory` in the docs example never sets it. Add a per-step timeout so
a single hung specialist can't stall a whole layer, and add a test that a step
sleeping past its budget is recorded as `failed` (not hung).

Good first issue because: the machinery exists (`call_with_timeout`,
`RetryPolicy`), the change is ~10 lines, and it comes with a clear test.
Pointers: orchestra/reliability.py (`Reliable`, `Timeout`), the L82 integration
cell.
"""

CI_YML = """name: ci
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e . pytest build twine
      - run: pytest -q
      - run: python -m build
      - run: twine check dist/*
"""

RELEASE_YML = """name: release
on:
  push:
    tags: ["v*"]
permissions:
  id-token: write        # trusted publishing to PyPI (no long-lived token)
jobs:
  publish:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install build
      - run: python -m build
      - uses: pypa/gh-action-pypi-publish@release/v1
"""

os.makedirs(os.path.join(PROJ, ".github", "workflows"), exist_ok=True)
writes = {
    "LAUNCH_CHECKLIST.md": LAUNCH_CHECKLIST,
    "first_issue.md": FIRST_ISSUE,
    ".github/workflows/ci.yml": CI_YML,
    ".github/workflows/release.yml": RELEASE_YML,
}
for rel, txt in writes.items():
    with open(os.path.join(PROJ, rel), "w") as f:
        f.write(txt)
    print("wrote", rel)

check("all four launch files were written",
      all(os.path.exists(os.path.join(PROJ, r)) for r in writes))
check("release workflow uses OIDC trusted publishing (no stored token)",
      "id-token: write" in RELEASE_YML and "password:" not in RELEASE_YML)

## §7 · The launch runbook (yours to authorize)

This is the sequence to run on your own machine when you decide to ship. It's
copy-paste, but it is **not** run here — publishing is your call, not the
notebook's.

```bash
# 0. one-time: create the GitHub repo `orchestra-agents`, add an MIT LICENSE
# 1. build + verify locally
python -m build
twine check dist/*
python -m venv /tmp/v && /tmp/v/bin/pip install dist/*.whl
/tmp/v/bin/python -c "import orchestra; print(orchestra.__version__)"

# 2. push, let CI go green on 3.10 / 3.11 / 3.12
git add . && git commit -m "orchestra-agents v0.1.0" && git push

# 3. ship: tag triggers the release workflow (trusted publishing to PyPI)
git tag v0.1.0 && git push --tags

# 4. after it lands: cross-link the sibling repos, pin the good-first-issue,
#    post the write-up, submit to one new awesome-list
```

## §8 · Ten ways a capstone ships broken

| # | Pitfall | The fix |
|---|---|---|
| 1 | Modules pass their own tests; no test runs them *together* | the load-bearing integration cell (§2) |
| 2 | Import name assumed == distribution name | `orchestra` (import) vs `orchestra-agents` (dist), stated in README |
| 3 | `pyproject` references a README written in a *later* cell → build fails | write every referenced file before the build step |
| 4 | "It imports" — but only from the source tree you've had on `sys.path` | install the built **wheel** into an isolated dir, import in a clean subprocess (§5) |
| 5 | Symbol collisions across modules silently shadow each other | static collision check (§1) — here, zero |
| 6 | A dead dependency crashes the whole plan | reliability layer + `best_effort` → `degraded`, not an exception (§3) |
| 7 | The generalist *fabricates* an answer it can't produce | honest fallback returns `None` → the step is honestly `failed` |
| 8 | Release workflow stores a long-lived PyPI token | OIDC trusted publishing (`id-token: write`) |
| 9 | CI tests one Python version, users hit another | matrix 3.10 / 3.11 / 3.12 |
| 10 | Four repos that link each other but don't connect in code | cross-promote *because* orchestra is observed by agent-obs, evaluated by agent-bench |

## §9 · Verification — every claim in one place

In [ ]:
passed = sum(1 for _, ok in CHECKS if ok)
print("Verification: %d/%d checks passed\n" % (passed, len(CHECKS)))
for name, ok in CHECKS:
    print(("  PASS" if ok else "  FAIL"), "-", name)
assert passed == len(CHECKS), "some checks failed: %s" % [n for n, ok in CHECKS if not ok]
print("\nALL %d CHECKS PASSED — orchestra-agents composes, builds, and installs." % len(CHECKS))

## Summary — and the end of Phase 9

| Capstone step | What it proved |
|---|---|
| Consolidate | five modules → one package with a single public API (25 symbols, zero collisions) |
| Integration | `ResilientOrchestrator` drives **all five pillars** in one `execute()` |
| Graceful degradation | a dead dependency → `degraded`, good answers preserved, nothing crashes |
| Build | `python -m build` → wheel + sdist; `twine check` PASSED |
| Fresh-wheel | the built artifact imports in a clean subprocess (a stranger can install it) |
| Launch | a coherent 4th OSS artifact with CI, trusted-publishing release, on-ramp issue |

**Phase 9 is complete (L77–L82, 6 lessons).** You went from "how do two agents
talk" to "a packaged, tested, installable multi-agent toolkit," and you now own
**four** cross-referencing open-source tools:

> **paper-distiller** (an agent) · **agent-bench** (offline eval) · **agent-obs**
> (production ops) · **orchestra** (multi-agent orchestration).

### Homework (pick one or two)
1. **Make the integration a real test.** Move §2/§3 into `tests/test_integration.py`
   as `pytest` cases and wire them into `ci.yml` so a regression fails the build.
2. **Per-step timeout.** Implement the good-first-issue: give `reliable_factory`
   a `timeout_s` and prove a step that sleeps past its budget is recorded
   `failed`, not hung.
3. **Observe it.** Wire `agent-obs`: emit one span per orchestrator step with
   `{status, attempts, breaker_state}`, then query the most-replanned category.
4. **Evaluate it.** Point `agent-bench` at the orchestrator: generate 40 compound
   goals, score completeness, and gate CI on a minimum score.
5. **A blackboard-native orchestrator.** Replace the plan-driven loop with
   `KnowledgeSource`s + `run_until_quiescent` (L78) so the orchestration is fully
   data-driven, and compare the two designs on the same goals.

### Phase 10 preview
With four shipped artifacts, the next arc is about depth rather than breadth.
Three candidate tracks (default **10A** unless you steer otherwise on the next
run):

- **10A · Retrieval-Augmented Generation at production scale** — chunking,
  embeddings, vector stores, hybrid search, reranking, and *evaluating* RAG
  (faithfulness, context precision/recall) with the agent-bench discipline.
- **10B · Fine-tuning & post-training** — when to fine-tune vs prompt, SFT,
  preference optimization (DPO), LoRA/QLoRA, and measuring lift honestly.
- **10C · Evaluation-driven agent development** — turn the whole portfolio into
  a single eval-first workflow: every change gated by offline eval + online
  guardrails.

Bring any question on the next run and I'll fold it into the curriculum before
we start 10A.